## Baseline QA using Samples Snapshot 1 Queries

First-pass lexical RAG baseline on top of the processed LongEval 2026 corpus. Using synthetic Task 4-style queries, the system retrieves top candidate documents with BM25, extracts the highest-overlap supporting sentences, and forms a grounded answer from the top evidence spans. This gives us a simple baseline to compare against stronger NLI and inference-based pipelines.

In [16]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00


In [1]:
!pip install rank-bm25 pandas pyarrow nltk

In [2]:
import pandas as pd
import pyarrow.parquet as pq
import json
import re
from rank_bm25 import BM25Okapi
from collections import Counter
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
parquet_path = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/longeval_documents.parquet"
pf = pq.ParquetFile(parquet_path)

pieces = []
rows_per_group = 300   # 8 * 300 = 2400 docs, manageable for a baseline demo

for rg in range(pf.num_row_groups):
    t = pf.read_row_group(rg, columns=["doc_id", "original_text", "metadata"])
    pieces.append(t.slice(0, rows_per_group).to_pandas())

docs_df = pd.concat(pieces, ignore_index=True)
print(docs_df.shape)
docs_df.head(2)

(2400, 3)


,doc_id,original_text,metadata
0,8724,Two new species of Pterostichus Bonelli subgen...,"{""arxivId"":null,""authors"":[{""name"":""Bergdahl,J..."
1,8725,When to sample in an inaccessible landscape: a...,"{""arxivId"":null,""authors"":[{""name"":""Harry, Ing..."


In [5]:
def normalize_text(x):
    x = str(x or "")
    x = re.sub(r"\s+", " ", x).strip()
    return x

docs_df["doc_id"] = docs_df["doc_id"].astype(str)
docs_df["original_text"] = docs_df["original_text"].apply(normalize_text)

doc_lookup = dict(zip(docs_df["doc_id"], docs_df["original_text"]))

docs_df = docs_df[docs_df["original_text"].str.len() > 100].reset_index(drop=True)
print(docs_df.shape)

(2400, 3)


In [6]:
queries_path = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/synthetic_queries_500.csv"
queries_df = pd.read_csv(queries_path)
queries_df["query_id"] = queries_df["query_id"].astype(str)
queries_df["anchor_doc_id"] = queries_df["anchor_doc_id"].astype(str)

def parse_candidate_doc_ids(x):
    if pd.isna(x) or not str(x).strip():
        return []
    return [s.strip() for s in str(x).split("|") if s.strip()]

queries_df["candidate_doc_ids"] = queries_df["candidate_doc_ids"].apply(parse_candidate_doc_ids)
queries_df.head(3)

,query_id,anchor_doc_id,topic,category,query,candidate_doc_ids
0,1,17225098,Asymptotic Behavior of the Isotropic-Nematic a...,challenges,What tradeoffs are reported for Asymptotic Beh...,"[17225098, 17224633, 17223989, 17226271, 17225..."
1,2,9395,Three new species in the harvestmen genus Acuc...,evidence,What results support Three new species in the ...,"[9395, 10107, 17224989, 10514, 9530, 292284320..."
2,3,168175649,Graph attention-driven document image classifi...,mechanisms,What processes underlie Graph attention-driven...,"[168175649, 168175407, 17225888, 287344299, 16..."


In [7]:
def get_candidate_docs(candidate_ids, doc_lookup):
    rows = []
    for doc_id in candidate_ids:
        text = doc_lookup.get(str(doc_id))
        if text:
            rows.append({"doc_id": str(doc_id), "original_text": text})
    return pd.DataFrame(rows)

In [8]:
from collections import Counter
from nltk.tokenize import sent_tokenize

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def lexical_overlap_score(query, sentence):
    q = Counter(tokenize(query))
    s = Counter(tokenize(sentence))
    if not q:
        return 0.0
    overlap = sum(min(q[t], s[t]) for t in q)
    return overlap / sum(q.values())

def extract_top_sentences_from_candidates(query, candidate_docs_df, max_extracts=5):
    candidates = []

    for _, row in candidate_docs_df.iterrows():
        doc_id = row["doc_id"]
        text = row["original_text"]

        for sent in sent_tokenize(text):
            sent = sent.strip()
            if len(sent) < 40:
                continue
            score = lexical_overlap_score(query, sent)
            candidates.append((score, doc_id, sent))

    candidates.sort(key=lambda x: x[0], reverse=True)

    seen = set()
    outputs = []
    for score, doc_id, sent in candidates:
        key = sent.lower()
        if key in seen:
            continue
        seen.add(key)
        outputs.append((doc_id, sent, score))
        if len(outputs) >= max_extracts:
            break

    return outputs

In [9]:
def lexical_overlap_score(query, sentence):
    q = Counter(tokenize(query))
    s = Counter(tokenize(sentence))
    if not q:
        return 0.0
    overlap = sum(min(q[t], s[t]) for t in q)
    return overlap / sum(q.values())

def extract_top_sentences(query, retrieved_docs, max_extracts=5):
    candidates = []

    for _, row in retrieved_docs.iterrows():
        doc_id = row["doc_id"]
        text = row["original_text"]

        for sent in sent_tokenize(text):
            sent = sent.strip()
            if len(sent) < 40:
                continue
            score = lexical_overlap_score(query, sent)
            candidates.append((score, doc_id, sent))

    candidates.sort(key=lambda x: x[0], reverse=True)

    seen = set()
    outputs = []
    for score, doc_id, sent in candidates:
        key = sent.lower()
        if key in seen:
            continue
        seen.add(key)
        outputs.append((doc_id, sent, score))
        if len(outputs) >= max_extracts:
            break

    return outputs

In [10]:
def make_baseline_answer(query, extracts):
    if not extracts:
        return "Insufficient evidence in retrieved extracts."

    top_sentences = [sent for _, sent, _ in extracts[:3]]
    answer = " ".join(top_sentences)

    # light cleanup
    answer = re.sub(r"\s+", " ", answer).strip()
    if len(answer) > 500:
        answer = answer[:500].rsplit(" ", 1)[0] + "..."

    return answer

In [18]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [27]:
import gc
import re
import torch

def shorten_extract(text, max_chars=220):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text[:max_chars]

def make_answer_prompt(query, extracts):
    trimmed = [shorten_extract(x, 260) for x in extracts[:3]]
    joined = "\n".join([f'[{i+1}] {x}' for i, x in enumerate(trimmed)])

    return f"""Answer the question using only the evidence below.

Instructions:
- Write 1-2 sentences.
- Do not restate the question.
- Summarize the main finding, process, or result from the evidence.
- If the evidence only gives the title/topic and not the answer, say exactly:
Insufficient evidence in retrieved extracts.

Question: {query}

Evidence:
{joined}

Answer:"""

def generate_answer_with_llm(query, extracts, max_new_tokens=256):
    if not extracts:
        return "Insufficient evidence in retrieved extracts."

    prompt = make_answer_prompt(query, extracts)

    with torch.inference_mode():
        out = generator(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_full_text=False,
            pad_token_id=tokenizer.eos_token_id,
        )[0]["generated_text"].strip()

    gc.collect()
    torch.cuda.empty_cache()

    return out if out else "Insufficient evidence in retrieved extracts."

In [28]:
baseline_rows_a = []
baseline_rows_b = []

test_queries = queries_df.head(50)   # start smaller first

for idx, (_, qrow) in enumerate(test_queries.iterrows()):
    qid = qrow["query_id"]
    query = qrow["query"]
    candidate_ids = qrow["candidate_doc_ids"]

    candidate_docs_df = get_candidate_docs(candidate_ids, doc_lookup)
    extracts = extract_top_sentences_from_candidates(query, candidate_docs_df, max_extracts=3)

    extract_texts = [shorten_extract(sent, 220) for _, sent, _ in extracts]

    try:
        answer = generate_answer_with_llm(query, extract_texts, max_new_tokens=56)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        answer = make_baseline_answer(query, extracts)

    baseline_rows_a.append({
        "query_id": qid,
        "answer": answer,
    })

    baseline_rows_b.append({
        "query_id": qid,
        "extract_1": extract_texts[0] if len(extract_texts) > 0 else "",
        "extract_2": extract_texts[1] if len(extract_texts) > 1 else "",
        "extract_3": extract_texts[2] if len(extract_texts) > 2 else "",
        "extract_4": "",
        "extract_5": "",
    })

    if idx % 5 == 0:
        print(f"finished {idx+1}/{len(test_queries)}")
        gc.collect()
        torch.cuda.empty_cache()

baseline_a_df = pd.DataFrame(baseline_rows_a)
baseline_b_df = pd.DataFrame(baseline_rows_b)

Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


finished 1/50


Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


finished 6/50


Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

...

Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

finished 41/50


Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


finished 46/50


Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=56) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [30]:
for i in range(min(10, len(baseline_a_df))):
    print("QUERY ID:", baseline_a_df.loc[i, "query_id"])
    print("QUERY:", test_queries.iloc[i]["query"])
    print("ANSWER:", baseline_a_df.loc[i, "answer"])
    print("EXTRACT 1:", baseline_b_df.loc[i, "extract_1"])
    print("EXTRACT 2:", baseline_b_df.loc[i, "extract_2"])
    print("EXTRACT 3:", baseline_b_df.loc[i, "extract_3"])
    print("=" * 120)
    print(" " * 120)

QUERY ID: 1
QUERY: What tradeoffs are reported for Asymptotic Behavior of the Isotropic-Nematic and Nematic-Columnar Phase Boundaries for?
ANSWER: [1] - No information provided about tradeoffs.
[2] - Focus is on asymptotic behavior without mentioning tradeoffs.
[3] - Heuristic argument suggests no significant tradeoffs between I-N and N-C phases. Insufficient evidence to determine specific tradeoffs.
EXTRACT 1: Asymptotic Behavior of the Isotropic-Nematic and Nematic-Columnar Phase Boundaries for the System of Hard Rectangles on a Square lattice A system of hard rectangles of size $m\times mk$ on a square lattice undergoes thre
EXTRACT 2: sta t-m ec h] 1 6 S ep 20 14 Asymptotic Behavior of the Isotropic-Nematic and Nematic-Columnar Phase Boundaries for the System of Hard Rectangles on a Square lattice Joyjit Kundu∗ and R. Rajesh† The Institute of Mathemat
EXTRACT 3: In this paper, we focus on the asymptotic behavior of the isotropic-nematic (I-N) and nematic-columnar (N-C) phase bounda

In [23]:
out_a = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/baseline_part_a.csv"
out_b = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/baseline_part_b.csv"

baseline_a_df.to_csv(out_a, index=False)
baseline_b_df.to_csv(out_b, index=False)

print("Saved:", out_a)
print("Saved:", out_b)

Saved: /content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/baseline_part_a.csv
Saved: /content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/baseline_part_b.csv
